# Agent From Scratch

## Objective

Build a simple Agentic AI system using pure Python without
LangChain, LangGraph, or CrewAI.

The agent will:

- Decide which tool to use
- Call tools using structured arguments
- Observe tool results
- Continue or stop based on the result
- Log Thought / Action / Observation
- Validate tool arguments using Pydantic
- Handle tool errors gracefully
- Retry failed tool calls
- Stop after a maximum number of steps
- Respect execution timeouts

## Tools

1. Calculator
2. Document Lookup
3. Mock Database Query

## 1. Imports

We first import the Python libraries required for the agent.

We will use:

- `time` for timeout and execution tracking
- `json` for structured tool arguments
- `typing` for type hints
- `pydantic` for argument validation
- `openai` for communicating with the LLM

In [3]:
import json
import time
from typing import Any, Dict

from pydantic import BaseModel, Field, ValidationError
from dotenv import load_dotenv
from openai import OpenAI

## 2. LLM Configuration

The agent needs an LLM to decide:

- which tool to use
- what arguments to provide
- whether to continue
- when to provide the final answer

The tools themselves are normal Python functions.

In [5]:
import os
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is not set.")

client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "openai/gpt-oss-20b"

print("LLM client configured successfully.")

LLM client configured successfully.


## 3. Tool Architecture

Each tool has four important parts:

1. Tool name
2. Tool description
3. Argument schema
4. Python implementation

Example:

    calculator
        ↓
    CalculatorArgs
        ↓
    calculate()

The LLM selects the tool, but Python performs the actual operation.

Pydantic validates the arguments before the function executes.

## 4. Calculator Tool

The calculator accepts one argument:

- `expression`

Pydantic will validate that the expression is a string.

In [6]:
class CalculatorArgs(BaseModel):
    expression: str = Field(
        ...,
        description="Mathematical expression to calculate"
    )

In [7]:
args = CalculatorArgs(
    expression="25 * 47"
)

print(args)

expression='25 * 47'


### Calculator Implementation

The LLM does not calculate the result itself through the tool.

It provides an expression, and our Python tool executes it.

For this learning exercise, we will keep the calculator simple.

In [8]:
def calculator(expression: str) -> float:
    """
    Calculate a simple mathematical expression.
    """
    allowed_chars = "0123456789+-*/(). "

    if not all(char in allowed_chars for char in expression):
        raise ValueError("Expression contains unsupported characters.")

    return eval(expression, {"__builtins__": {}}, {})

In [9]:
result = calculator("25 * 47")

print("Result:", result)

Result: 1175


## 5. Document Lookup Tool

The document lookup tool simulates a simple knowledge base.

The agent will provide a search query.

The tool will return relevant document content.

This is intentionally simple. It is not a vector database or RAG
system yet.

In [10]:
DOCUMENTS = {
    "docuchat": """
    DocuChat is a RAG-powered chat application.
    It uses PostgreSQL with pgvector to store document embeddings.
    Relevant document chunks are retrieved using semantic similarity.
    """,

    "rag": """
    Retrieval-Augmented Generation combines document retrieval
    with language model generation. Relevant context is retrieved
    before the answer is generated.
    """,

    "agents": """
    An AI agent can decide which action to take, use tools,
    observe results, and continue until the task is complete.
    """
}

In [11]:
class DocumentLookupArgs(BaseModel):
    query: str = Field(
        ...,
        min_length=1,
        description="Search query for the document store"
    )

In [12]:
args = DocumentLookupArgs(
    query="What does DocuChat use for embeddings?"
)

print(args)

query='What does DocuChat use for embeddings?'


In [13]:
def document_lookup(query: str) -> str:
    """
    Simple keyword-based document lookup.
    """

    query_lower = query.lower()

    matches = []

    for name, content in DOCUMENTS.items():
        if name in query_lower:
            matches.append(content.strip())

    if not matches:
        return "No relevant document found."

    return "\n\n".join(matches)

In [14]:
result = document_lookup("Tell me about DocuChat")

print(result)

DocuChat is a RAG-powered chat application.
    It uses PostgreSQL with pgvector to store document embeddings.
    Relevant document chunks are retrieved using semantic similarity.


## 6. Mock Database Query Tool

The third tool simulates a database.

The agent can ask for information such as:

- user information
- document information
- conversation information

This allows us to practice tool calling without requiring
a real database connection.

In [15]:
MOCK_DB = {
    "users": [
        {
            "id": 1,
            "name": "Arun",
            "email": "arun@example.com"
        },
        {
            "id": 2,
            "name": "Raj",
            "email": "raj@example.com"
        }
    ],
    "documents": [
        {
            "id": 1,
            "filename": "docuchat.txt",
            "chunks": 5
        },
        {
            "id": 2,
            "filename": "rag_notes.txt",
            "chunks": 8
        }
    ]
}

In [16]:
class MockDBArgs(BaseModel):
    table: str = Field(
        ...,
        description="Table to query"
    )

    limit: int = Field(
        default=5,
        ge=1,
        le=20,
        description="Maximum number of rows to return"
    )

In [17]:
def mock_db_query(table: str, limit: int = 5) -> list:
    """
    Query the mock database.
    """

    if table not in MOCK_DB:
        raise ValueError(
            f"Unknown table: {table}"
        )

    return MOCK_DB[table][:limit]

In [18]:
result = mock_db_query("users", 2)

print(result)

[{'id': 1, 'name': 'Arun', 'email': 'arun@example.com'}, {'id': 2, 'name': 'Raj', 'email': 'raj@example.com'}]


## 7. Pydantic Argument Validation

The LLM may generate incorrect tool arguments.

We therefore validate arguments before executing a tool.

Invalid arguments should never directly reach the tool implementation.

In [19]:
try:
    args = MockDBArgs(
        table="users",
        limit=100
    )

except ValidationError as e:
    print("Validation failed:")
    print(e)

Validation failed:
1 validation error for MockDBArgs
limit
  Input should be less than or equal to 20 [type=less_than_equal, input_value=100, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


## 8. Tool Registry

The tool registry connects:

- tool name
- description
- Pydantic argument model
- Python implementation

In [21]:
TOOLS = {
    "calculator": {
        "description": "Calculate a mathematical expression.",
        "args_model": CalculatorArgs,
        "function": calculator
    },

    "document_lookup": {
        "description": "Search the document knowledge base.",
        "args_model": DocumentLookupArgs,
        "function": document_lookup
    },

    "mock_db_query": {
        "description": "Query the mock database.",
        "args_model": MockDBArgs,
        "function": mock_db_query
    }
}

In [22]:
def execute_tool(tool_name: str, arguments: Dict[str, Any]):
    """
    Validate arguments and execute the requested tool.
    """

    if tool_name not in TOOLS:
        raise ValueError(
            f"Unknown tool: {tool_name}"
        )

    tool = TOOLS[tool_name]

    args_model = tool["args_model"]
    function = tool["function"]

    validated_args = args_model.model_validate(arguments)

    result = function(**validated_args.model_dump())

    return result

In [23]:
result = execute_tool(
    "calculator",
    {"expression": "100 / 4"}
)

print(result)

25.0


## 9. Agent Logging

Every agent step will record:

### Thought
Why the agent decided to take an action.

### Action
Which tool it selected and the arguments.

### Observation
What the tool returned.

Example:

Thought:
"I need to calculate 25 * 47."

Action:
calculator({"expression": "25 * 47"})

Observation:
1175

In [24]:
def log_event(event_type: str, content: Any):
    print(f"\n[{event_type}]")
    print(content)

In [25]:
log_event("Thought", "I need to calculate 25 * 47.")
log_event("Action", {
    "tool": "calculator",
    "arguments": {
        "expression": "25 * 47"
    }
})
log_event("Observation", 1175)


[Thought]
I need to calculate 25 * 47.

[Action]
{'tool': 'calculator', 'arguments': {'expression': '25 * 47'}}

[Observation]
1175


## 10. Agent Decision Prompt

The LLM is instructed to:

1. Understand the user's request.
2. Decide whether a tool is required.
3. Select one of the available tools.
4. Provide valid JSON arguments.
5. Use observations from previous tool calls.
6. Return a final answer when the task is complete.

In [48]:
SYSTEM_PROMPT = """
You are a simple tool-using AI agent.

Your job is to solve the user's request using the available tools.

Available tools:
- calculator
- document_lookup
- mock_db_query

Use a tool when it is necessary.
After receiving a tool result, continue solving the user's original request.

Do not invent tool results.
Do not invent tools.
If the task can be answered without a tool, answer directly.
"""

In [55]:
def ask_agent(messages):
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0,
        reasoning_effort="low",
        tools=[
            {
                "type": "function",
                "function": {
                    "name": "calculator",
                    "description": "Calculate a mathematical expression.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "expression": {
                                "type": "string",
                                "description": "Mathematical expression to calculate"
                            }
                        },
                        "required": ["expression"],
                        "additionalProperties": False
                    }
                }
            },
            {
                "type": "function",
                "function": {
                    "name": "document_lookup",
                    "description": "Search the document knowledge base.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "query": {
                                "type": "string",
                                "description": "Search query"
                            }
                        },
                        "required": ["query"],
                        "additionalProperties": False
                    }
                }
            },
            {
                "type": "function",
                "function": {
                    "name": "mock_db_query",
                    "description": "Query the mock database.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "table": {
                                "type": "string",
                                "description": "Database table"
                            },
                            "limit": {
                                "type": "integer",
                                "minimum": 1,
                                "maximum": 20,
                                "description": "Maximum number of rows"
                            }
                        },
                        "required": ["table", "limit"],
                        "additionalProperties": False
                    }
                }
            }
        ],
        tool_choice="auto"
    )

    return response

In [65]:
def extract_tool_call(response):
    message = response.choices[0].message

    if not message.tool_calls:
        return None

    tool_call = message.tool_calls[0]

    return {
        "id": tool_call.id,
        "tool": tool_call.function.name,
        "arguments": json.loads(tool_call.function.arguments)
    }

In [66]:
MAX_RETRIES = 2
MAX_STEPS = 8
TIMEOUT_SECONDS = 60

In [67]:
def safe_execute_tool(
    tool_name: str,
    arguments: Dict[str, Any]
):
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):

        try:
            log_event(
                "Tool Attempt",
                f"{tool_name} - attempt {attempt}"
            )

            result = execute_tool(
                tool_name,
                arguments
            )

            return {
                "success": True,
                "result": result
            }

        except ValidationError as e:
            return {
                "success": False,
                "error": f"Argument validation failed: {e}"
            }

        except Exception as e:
            last_error = str(e)

            log_event(
                "Tool Error",
                last_error
            )

            if attempt < MAX_RETRIES:
                time.sleep(0.5)

    return {
        "success": False,
        "error": last_error
    }

In [68]:
messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": "What is 125 * 48?"
    }
]

response = ask_agent(messages)

print("CONTENT:")
print(response.choices[0].message.content)

print("\nTOOL CALLS:")
print(response.choices[0].message.tool_calls)

CONTENT:
None

TOOL CALLS:
[ChatCompletionMessageFunctionToolCall(id='fc_e1a1ee7f-12ee-4456-86b3-9ca901eef0c6', function=Function(arguments='{"expression":"125 * 48"}', name='calculator'), type='function')]


In [69]:
tool_call = response.choices[0].message.tool_calls[0]

tool_name = tool_call.function.name
arguments = json.loads(tool_call.function.arguments)

print("Tool name:", tool_name)
print("Arguments:", arguments)

Tool name: calculator
Arguments: {'expression': '125 * 48'}


In [70]:
result = execute_tool(tool_name, arguments)

print("Tool result:", result)

Tool result: 6000


In [71]:
try:
    result = execute_tool(
        "calculator",
        {
            "wrong_argument": "125 * 48"
        }
    )

    print(result)

except ValidationError as e:
    print("VALIDATION FAILED")
    print(e)

VALIDATION FAILED
1 validation error for CalculatorArgs
expression
  Field required [type=missing, input_value={'wrong_argument': '125 * 48'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


In [73]:
def log_tool_action(tool_name, arguments):
    log_event(
        "Action",
        {
            "tool": tool_name,
            "arguments": arguments
        }
    )

In [75]:
def run_agent(user_request: str):
    start_time = time.time()

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_request
        }
    ]

    for step in range(1, MAX_STEPS + 1):

        log_event("Step", step)

        # Timeout
        if time.time() - start_time >= TIMEOUT_SECONDS:
            log_event("Stop", "Timeout reached.")
            return "Agent stopped because the timeout was reached."

        # Ask LLM
        try:
            response = ask_agent(messages)
        except Exception as e:
            log_event("LLM Error", str(e))
            return "Agent stopped because the LLM request failed."

        message = response.choices[0].message

        # No tool call means final answer
        if not message.tool_calls:
            final_answer = message.content or "No final answer provided."
            log_event("Final", final_answer)
            return final_answer

        # Get tool call
        tool_call = message.tool_calls[0]
        tool_name = tool_call.function.name

        # Parse tool arguments
        try:
            arguments = json.loads(tool_call.function.arguments)
        except json.JSONDecodeError:
            log_event("Tool Error", "Invalid tool arguments.")
            return "Agent stopped because the tool arguments were invalid."

        # Thought
        log_event(
            "Thought",
            f"Agent decided to use the '{tool_name}' tool."
        )

        # Action
        log_event(
            "Action",
            {
                "tool": tool_name,
                "arguments": arguments
            }
        )

        # Execute tool
        tool_result = safe_execute_tool(
            tool_name,
            arguments
        )

        # Observation
        log_event(
            "Observation",
            tool_result
        )

        # Add assistant tool-call message
        messages.append(message)

        # Add tool result
        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(
                    tool_result,
                    default=str
                )
            }
        )

    log_event(
        "Stop",
        "Maximum number of steps reached."
    )

    return "Agent stopped because the maximum number of steps was reached."

In [76]:
import inspect

print(inspect.getsource(run_agent))

def run_agent(user_request: str):
    start_time = time.time()

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_request
        }
    ]

    for step in range(1, MAX_STEPS + 1):

        log_event("Step", step)

        # Timeout
        if time.time() - start_time >= TIMEOUT_SECONDS:
            log_event("Stop", "Timeout reached.")
            return "Agent stopped because the timeout was reached."

        # Ask LLM
        try:
            response = ask_agent(messages)
        except Exception as e:
            log_event("LLM Error", str(e))
            return "Agent stopped because the LLM request failed."

        message = response.choices[0].message

        # No tool call means final answer
        if not message.tool_calls:
            final_answer = message.content or "No final answer provided."
            log_event("Final", final_answer)
            r

In [77]:
result = run_agent(
    "What does DocuChat use to store document embeddings?"
)

print("\nFINAL RESULT:")
print(result)


[Step]
1

[Thought]
Agent decided to use the 'document_lookup' tool.

[Action]
{'tool': 'document_lookup', 'arguments': {'query': 'DocuChat store document embeddings'}}

[Tool Attempt]
document_lookup - attempt 1

[Observation]
{'success': True, 'result': 'DocuChat is a RAG-powered chat application.\n    It uses PostgreSQL with pgvector to store document embeddings.\n    Relevant document chunks are retrieved using semantic similarity.'}

[Step]
2

[Final]
DocuChat stores document embeddings in **PostgreSQL using the pgvector extension**. This allows the application to keep the embeddings in a relational database while still supporting efficient similarity searches.

FINAL RESULT:
DocuChat stores document embeddings in **PostgreSQL using the pgvector extension**. This allows the application to keep the embeddings in a relational database while still supporting efficient similarity searches.


In [78]:
result = run_agent(
    "Show me the users in the database."
)

print("\nFINAL RESULT:")
print(result)


[Step]
1

[Thought]
Agent decided to use the 'mock_db_query' tool.

[Action]
{'tool': 'mock_db_query', 'arguments': {'limit': 20, 'table': 'users'}}

[Tool Attempt]
mock_db_query - attempt 1

[Observation]
{'success': True, 'result': [{'id': 1, 'name': 'Arun', 'email': 'arun@example.com'}, {'id': 2, 'name': 'Raj', 'email': 'raj@example.com'}]}

[Step]
2

[Final]
Here are the users currently in the database:

| ID | Name | Email |
|----|------|-------------------|
| 1  | Arun | arun@example.com |
| 2  | Raj  | raj@example.com |

Let me know if you need more details or anything else!

FINAL RESULT:
Here are the users currently in the database:

| ID | Name | Email |
|----|------|-------------------|
| 1  | Arun | arun@example.com |
| 2  | Raj  | raj@example.com |

Let me know if you need more details or anything else!


In [79]:
result = safe_execute_tool(
    "calculator",
    {
        "wrong_argument": "125 * 48"
    }
)

print("RESULT:")
print(result)


[Tool Attempt]
calculator - attempt 1
RESULT:
{'success': False, 'error': "Argument validation failed: 1 validation error for CalculatorArgs\nexpression\n  Field required [type=missing, input_value={'wrong_argument': '125 * 48'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing"}


In [80]:
result = safe_execute_tool(
    "mock_db_query",
    {
        "table": "passwords",
        "limit": 5
    }
)

print("RESULT:")
print(result)


[Tool Attempt]
mock_db_query - attempt 1

[Tool Error]
Unknown table: passwords

[Tool Attempt]
mock_db_query - attempt 2

[Tool Error]
Unknown table: passwords
RESULT:
{'success': False, 'error': 'Unknown table: passwords'}


In [81]:
# Step 33: Test maximum-step protection

MAX_STEPS = 2

result = run_agent(
    "Investigate DocuChat. Find what it uses for embeddings and also check the documents in the database."
)

print("\nFINAL RESULT:")
print(result)

# Restore normal limit
MAX_STEPS = 8


[Step]
1

[Thought]
Agent decided to use the 'document_lookup' tool.

[Action]
{'tool': 'document_lookup', 'arguments': {'query': 'DocuChat embeddings'}}

[Tool Attempt]
document_lookup - attempt 1

[Observation]
{'success': True, 'result': 'DocuChat is a RAG-powered chat application.\n    It uses PostgreSQL with pgvector to store document embeddings.\n    Relevant document chunks are retrieved using semantic similarity.'}

[Step]
2

[Thought]
Agent decided to use the 'mock_db_query' tool.

[Action]
{'tool': 'mock_db_query', 'arguments': {'limit': 5, 'table': 'documents'}}

[Tool Attempt]
mock_db_query - attempt 1

[Observation]
{'success': True, 'result': [{'id': 1, 'filename': 'docuchat.txt', 'chunks': 5}, {'id': 2, 'filename': 'rag_notes.txt', 'chunks': 8}]}

[Stop]
Maximum number of steps reached.

FINAL RESULT:
Agent stopped because the maximum number of steps was reached.


In [82]:
# Step 34: Test timeout protection

TIMEOUT_SECONDS = 0

result = run_agent(
    "What is 125 * 48?"
)

print("\nFINAL RESULT:")
print(result)

# Restore normal timeout
TIMEOUT_SECONDS = 60


[Step]
1

[Stop]
Timeout reached.

FINAL RESULT:
Agent stopped because the timeout was reached.


In [83]:
# Step 35: Final calculator test

MAX_STEPS = 8
TIMEOUT_SECONDS = 60

result = run_agent(
    "Calculate 125 * 48."
)

print("\nFINAL RESULT:")
print(result)


[Step]
1

[Thought]
Agent decided to use the 'calculator' tool.

[Action]
{'tool': 'calculator', 'arguments': {'expression': '125 * 48'}}

[Tool Attempt]
calculator - attempt 1

[Observation]
{'success': True, 'result': 6000}

[Step]
2

[Final]
125 × 48 = **6,000**

FINAL RESULT:
125 × 48 = **6,000**


In [84]:
# Step 36: Final document lookup test

result = run_agent(
    "What does DocuChat use to store document embeddings?"
)

print("\nFINAL RESULT:")
print(result)


[Step]
1

[Thought]
Agent decided to use the 'document_lookup' tool.

[Action]
{'tool': 'document_lookup', 'arguments': {'query': 'DocuChat store document embeddings'}}

[Tool Attempt]
document_lookup - attempt 1

[Observation]
{'success': True, 'result': 'DocuChat is a RAG-powered chat application.\n    It uses PostgreSQL with pgvector to store document embeddings.\n    Relevant document chunks are retrieved using semantic similarity.'}

[Step]
2

[Final]
DocuChat stores document embeddings in **PostgreSQL using the pgvector extension**. This allows the application to keep the embeddings in a relational database while still supporting efficient similarity searches.

FINAL RESULT:
DocuChat stores document embeddings in **PostgreSQL using the pgvector extension**. This allows the application to keep the embeddings in a relational database while still supporting efficient similarity searches.


In [85]:
# Step 37: Final mock database test

result = run_agent(
    "Show me the users stored in the database."
)

print("\nFINAL RESULT:")
print(result)


[Step]
1

[Thought]
Agent decided to use the 'mock_db_query' tool.

[Action]
{'tool': 'mock_db_query', 'arguments': {'limit': 20, 'table': 'users'}}

[Tool Attempt]
mock_db_query - attempt 1

[Observation]
{'success': True, 'result': [{'id': 1, 'name': 'Arun', 'email': 'arun@example.com'}, {'id': 2, 'name': 'Raj', 'email': 'raj@example.com'}]}

[Step]
2

[Final]
Here are the users currently stored in the database:

| ID | Name | Email |
|----|------|-------------------|
| 1  | Arun | arun@example.com |
| 2  | Raj  | raj@example.com |

Let me know if you need more details or anything else!

FINAL RESULT:
Here are the users currently stored in the database:

| ID | Name | Email |
|----|------|-------------------|
| 1  | Arun | arun@example.com |
| 2  | Raj  | raj@example.com |

Let me know if you need more details or anything else!


In [86]:
# Step 38: Test all three tools

test_requests = [
    "Calculate 25 * 47.",
    "What does DocuChat use to store document embeddings?",
    "Show me the users in the database."
]

for i, request in enumerate(test_requests, start=1):
    print("\n" + "=" * 80)
    print(f"TEST {i}")
    print("=" * 80)
    print("REQUEST:", request)

    result = run_agent(request)

    print("\nFINAL RESULT:")
    print(result)


TEST 1
REQUEST: Calculate 25 * 47.

[Step]
1

[Thought]
Agent decided to use the 'calculator' tool.

[Action]
{'tool': 'calculator', 'arguments': {'expression': '25 * 47'}}

[Tool Attempt]
calculator - attempt 1

[Observation]
{'success': True, 'result': 1175}

[Step]
2

[Final]
The result of \(25 \times 47\) is **1,175**.

FINAL RESULT:
The result of \(25 \times 47\) is **1,175**.

TEST 2
REQUEST: What does DocuChat use to store document embeddings?

[Step]
1

[Thought]
Agent decided to use the 'document_lookup' tool.

[Action]
{'tool': 'document_lookup', 'arguments': {'query': 'DocuChat store document embeddings'}}

[Tool Attempt]
document_lookup - attempt 1

[Observation]
{'success': True, 'result': 'DocuChat is a RAG-powered chat application.\n    It uses PostgreSQL with pgvector to store document embeddings.\n    Relevant document chunks are retrieved using semantic similarity.'}

[Step]
2

[Final]
DocuChat stores document embeddings in **PostgreSQL using the pgvector extension**

In [87]:
# Step 39B: Tool error + retry test

result = safe_execute_tool(
    "mock_db_query",
    {
        "table": "passwords",
        "limit": 5
    }
)

print("\nDATABASE ERROR TEST:")
print(result)


[Tool Attempt]
mock_db_query - attempt 1

[Tool Error]
Unknown table: passwords

[Tool Attempt]
mock_db_query - attempt 2

[Tool Error]
Unknown table: passwords

DATABASE ERROR TEST:
{'success': False, 'error': 'Unknown table: passwords'}


In [89]:


import time
import json
import concurrent.futures
import pandas as pd
from collections import Counter


# ============================================================
# 1. BAD vs GOOD TOOL DESCRIPTIONS
#    10 prompts -> tool-selection accuracy
# ============================================================

print("=" * 80)
print("EXPERIMENT 1: BAD vs GOOD TOOL DESCRIPTIONS")
print("=" * 80)


evaluation_prompts = [
    ("Calculate 125 * 48.", "calculator"),
    ("What is 25 + 75?", "calculator"),
    ("What does DocuChat use to store document embeddings?", "document_lookup"),
    ("Explain what RAG means based on the documents.", "document_lookup"),
    ("Show me the users in the database.", "mock_db_query"),
    ("List the documents stored in the database.", "mock_db_query"),
    ("Calculate 15 * 20.", "calculator"),
    ("Find information about AI agents in the documents.", "document_lookup"),
    ("Show me the documents in the database.", "mock_db_query"),
    ("Calculate 1000 / 25.", "calculator"),
]


bad_descriptions = {
    "calculator": "Does things with numbers.",
    "document_lookup": "Gets information.",
    "mock_db_query": "Gets database stuff."
}


good_descriptions = {
    "calculator": (
        "Calculate mathematical expressions such as addition, "
        "subtraction, multiplication, division, and arithmetic calculations."
    ),
    "document_lookup": (
        "Search the document knowledge base for information contained "
        "in DocuChat, RAG, AI agents, and stored documents."
    ),
    "mock_db_query": (
        "Query structured database tables such as users and documents. "
        "Use this when the user asks to show, list, count, or retrieve "
        "database records."
    )
}


def build_experiment_tools(descriptions):
    return [
        {
            "type": "function",
            "function": {
                "name": "calculator",
                "description": descriptions["calculator"],
                "parameters": {
                    "type": "object",
                    "properties": {
                        "expression": {
                            "type": "string"
                        }
                    },
                    "required": ["expression"],
                    "additionalProperties": False
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "document_lookup",
                "description": descriptions["document_lookup"],
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {
                            "type": "string"
                        }
                    },
                    "required": ["query"],
                    "additionalProperties": False
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "mock_db_query",
                "description": descriptions["mock_db_query"],
                "parameters": {
                    "type": "object",
                    "properties": {
                        "table": {
                            "type": "string"
                        },
                        "limit": {
                            "type": "integer",
                            "minimum": 1,
                            "maximum": 20
                        }
                    },
                    "required": ["table", "limit"],
                    "additionalProperties": False
                }
            }
        }
    ]


def evaluate_tool_selection(descriptions):
    results = []

    tools = build_experiment_tools(descriptions)

    for prompt, expected_tool in evaluation_prompts:

        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0,
                reasoning_effort="low",
                tools=tools,
                tool_choice="auto"
            )

            message = response.choices[0].message

            if message.tool_calls:
                selected_tool = message.tool_calls[0].function.name
            else:
                selected_tool = "none"

            correct = selected_tool == expected_tool

        except Exception as e:
            selected_tool = "error"
            correct = False

        results.append({
            "Prompt": prompt,
            "Expected Tool": expected_tool,
            "Selected Tool": selected_tool,
            "Correct": correct
        })

    return results


bad_tool_results = evaluate_tool_selection(
    bad_descriptions
)

good_tool_results = evaluate_tool_selection(
    good_descriptions
)


bad_accuracy = (
    sum(row["Correct"] for row in bad_tool_results)
    / len(bad_tool_results)
    * 100
)

good_accuracy = (
    sum(row["Correct"] for row in good_tool_results)
    / len(good_tool_results)
    * 100
)


tool_description_results = pd.DataFrame([
    {
        "Description": "Bad",
        "Correct": sum(
            row["Correct"]
            for row in bad_tool_results
        ),
        "Total Prompts": 10,
        "Accuracy (%)": round(bad_accuracy, 2)
    },
    {
        "Description": "Good",
        "Correct": sum(
            row["Correct"]
            for row in good_tool_results
        ),
        "Total Prompts": 10,
        "Accuracy (%)": round(good_accuracy, 2)
    }
])


print("\nTool-selection accuracy:")
display(tool_description_results)


# ============================================================
# 2. ReAct vs PLAN-AND-EXECUTE
#    Same 5 tasks -> steps, tokens, success
# ============================================================

print("\n" + "=" * 80)
print("EXPERIMENT 2: ReAct vs PLAN-AND-EXECUTE")
print("=" * 80)


comparison_tasks = [
    "Calculate 125 * 48.",
    "What does DocuChat use to store document embeddings?",
    "Show me the users in the database.",
    "Find information about RAG in the documents.",
    "List the documents stored in the database."
]


def measure_react(task):

    start_time = time.time()

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": task
        }
    ]

    total_tokens = 0

    for step in range(1, MAX_STEPS + 1):

        if time.time() - start_time >= TIMEOUT_SECONDS:
            return {
                "Steps": step,
                "Tokens": total_tokens,
                "Success": False,
                "Time (sec)": round(
                    time.time() - start_time, 3
                )
            }

        try:
            response = ask_agent(messages)

            if response.usage:
                total_tokens += response.usage.total_tokens

            message = response.choices[0].message

            if not message.tool_calls:
                return {
                    "Steps": step,
                    "Tokens": total_tokens,
                    "Success": bool(message.content),
                    "Time (sec)": round(
                        time.time() - start_time, 3
                    )
                }

            tool_call = message.tool_calls[0]

            arguments = json.loads(
                tool_call.function.arguments
            )

            tool_result = safe_execute_tool(
                tool_call.function.name,
                arguments
            )

            messages.append(message)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(
                    tool_result,
                    default=str
                )
            })

        except Exception:
            return {
                "Steps": step,
                "Tokens": total_tokens,
                "Success": False,
                "Time (sec)": round(
                    time.time() - start_time, 3
                )
            }

    return {
        "Steps": MAX_STEPS,
        "Tokens": total_tokens,
        "Success": False,
        "Time (sec)": round(
            time.time() - start_time, 3
        )
    }


def create_plan(task):

    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        reasoning_effort="low",
        messages=[
            {
                "role": "system",
                "content": """
You are a planning component.

Create a short plan for solving the user's task.

Available tools:
- calculator
- document_lookup
- mock_db_query

Return a concise numbered plan.
"""
            },
            {
                "role": "user",
                "content": task
            }
        ]
    )

    tokens = (
        response.usage.total_tokens
        if response.usage
        else 0
    )

    return response.choices[0].message.content, tokens


def measure_plan_execute(task):

    start_time = time.time()

    try:

        plan, planner_tokens = create_plan(task)

        messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"""
User task:
{task}

Plan:
{plan}

Execute the plan and provide the final answer.
"""
            }
        ]

        total_tokens = planner_tokens
        steps = 1

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0,
            reasoning_effort="low",
            tools=build_experiment_tools(
                good_descriptions
            ),
            tool_choice="auto"
        )

        if response.usage:
            total_tokens += response.usage.total_tokens

        steps += 1

        message = response.choices[0].message

        if message.tool_calls:

            messages.append(message)

            for tool_call in message.tool_calls:

                arguments = json.loads(
                    tool_call.function.arguments
                )

                tool_result = safe_execute_tool(
                    tool_call.function.name,
                    arguments
                )

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(
                        tool_result,
                        default=str
                    )
                })

            final_response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=0,
                reasoning_effort="low"
            )

            if final_response.usage:
                total_tokens += (
                    final_response.usage.total_tokens
                )

            steps += 1

            success = bool(
                final_response.choices[0].message.content
            )

        else:
            success = bool(message.content)

        return {
            "Steps": steps,
            "Tokens": total_tokens,
            "Success": success,
            "Time (sec)": round(
                time.time() - start_time, 3
            )
        }

    except Exception:
        return {
            "Steps": 0,
            "Tokens": 0,
            "Success": False,
            "Time (sec)": round(
                time.time() - start_time, 3
            )
        }


comparison_rows = []

for task in comparison_tasks:

    print("\nRunning:", task)

    react = measure_react(task)

    plan = measure_plan_execute(task)

    comparison_rows.append({
        "Task": task,

        "ReAct Steps": react["Steps"],
        "ReAct Tokens": react["Tokens"],
        "ReAct Success": react["Success"],

        "Plan Steps": plan["Steps"],
        "Plan Tokens": plan["Tokens"],
        "Plan Success": plan["Success"]
    })


react_plan_results = pd.DataFrame(
    comparison_rows
)


print("\nReAct vs Plan-and-Execute:")
display(react_plan_results)


# ============================================================
# 3. TEMPERATURE 0 vs 0.7
#    Run each 5 times
# ============================================================

print("\n" + "=" * 80)
print("EXPERIMENT 3: TEMPERATURE 0 vs 0.7")
print("=" * 80)


temperature_prompt = (
    "What does DocuChat use to store document embeddings?"
)


def temperature_test(temperature):

    results = []

    for run_number in range(1, 6):

        try:

            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT
                    },
                    {
                        "role": "user",
                        "content": temperature_prompt
                    }
                ],
                temperature=temperature,
                reasoning_effort="low",
                tools=build_experiment_tools(
                    good_descriptions
                ),
                tool_choice="auto"
            )

            message = response.choices[0].message

            tool = (
                message.tool_calls[0].function.name
                if message.tool_calls
                else "none"
            )

            answer = message.content or ""

            tokens = (
                response.usage.total_tokens
                if response.usage
                else 0
            )

            results.append({
                "Temperature": temperature,
                "Run": run_number,
                "Selected Tool": tool,
                "Answer": answer,
                "Tokens": tokens
            })

        except Exception as e:

            results.append({
                "Temperature": temperature,
                "Run": run_number,
                "Selected Tool": "error",
                "Answer": str(e),
                "Tokens": 0
            })

    return results


temperature_0 = temperature_test(0)

temperature_07 = temperature_test(0.7)


temperature_results = pd.DataFrame(
    temperature_0 + temperature_07
)


print("\nAll temperature runs:")
display(temperature_results)


def calculate_consistency(results):

    tools = [
        row["Selected Tool"]
        for row in results
    ]

    counts = Counter(tools)

    most_common_tool, count = (
        counts.most_common(1)[0]
    )

    return round(
        count / len(tools) * 100,
        2
    )


temperature_summary = pd.DataFrame([
    {
        "Temperature": 0,
        "Runs": 5,
        "Consistency (%)": calculate_consistency(
            temperature_0
        )
    },
    {
        "Temperature": 0.7,
        "Runs": 5,
        "Consistency (%)": calculate_consistency(
            temperature_07
        )
    }
])


print("\nTemperature consistency:")
display(temperature_summary)


# ============================================================
# 4. PARALLEL TOOL CALLS
#    Sequential vs parallel execution
# ============================================================

print("\n" + "=" * 80)
print("EXPERIMENT 4: PARALLEL TOOL CALLS")
print("=" * 80)


def sequential_execution():

    start_time = time.time()

    users = safe_execute_tool(
        "mock_db_query",
        {
            "table": "users",
            "limit": 5
        }
    )

    documents = safe_execute_tool(
        "mock_db_query",
        {
            "table": "documents",
            "limit": 5
        }
    )

    elapsed = time.time() - start_time

    return {
        "Mode": "Sequential",
        "Tool Calls": 2,
        "Time (sec)": round(elapsed, 6),
        "Success": (
            users["success"]
            and documents["success"]
        )
    }


def parallel_execution():

    start_time = time.time()

    calls = [
        (
            "mock_db_query",
            {
                "table": "users",
                "limit": 5
            }
        ),
        (
            "mock_db_query",
            {
                "table": "documents",
                "limit": 5
            }
        )
    ]

    with concurrent.futures.ThreadPoolExecutor(
        max_workers=2
    ) as executor:

        futures = [
            executor.submit(
                safe_execute_tool,
                tool_name,
                arguments
            )
            for tool_name, arguments in calls
        ]

        results = [
            future.result()
            for future in futures
        ]

    elapsed = time.time() - start_time

    return {
        "Mode": "Parallel",
        "Tool Calls": 2,
        "Time (sec)": round(elapsed, 6),
        "Success": all(
            result["success"]
            for result in results
        )
    }


sequential_result = sequential_execution()

parallel_result = parallel_execution()


parallel_results = pd.DataFrame([
    sequential_result,
    parallel_result
])


print("\nSequential vs Parallel:")
display(parallel_results)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("FINAL EXPERIMENT SUMMARY")
print("=" * 80)


react_success = sum(
    row["ReAct Success"]
    for row in comparison_rows
)

plan_success = sum(
    row["Plan Success"]
    for row in comparison_rows
)


summary = pd.DataFrame([
    {
        "Experiment": "Bad vs Good Tool Descriptions",
        "Measurement": "Tool-selection accuracy",
        "Result": (
            f"Bad: {bad_accuracy:.2f}% | "
            f"Good: {good_accuracy:.2f}%"
        )
    },
    {
        "Experiment": "ReAct vs Plan-and-Execute",
        "Measurement": "Success on 5 tasks",
        "Result": (
            f"ReAct: {react_success}/5 | "
            f"Plan: {plan_success}/5"
        )
    },
    {
        "Experiment": "Temperature 0 vs 0.7",
        "Measurement": "Consistency over 5 runs",
        "Result": (
            f"T=0: "
            f"{temperature_summary.iloc[0]['Consistency (%)']}% | "
            f"T=0.7: "
            f"{temperature_summary.iloc[1]['Consistency (%)']}%"
        )
    },
    {
        "Experiment": "Parallel Tool Calls",
        "Measurement": "Execution time",
        "Result": (
            f"Sequential: "
            f"{sequential_result['Time (sec)']} sec | "
            f"Parallel: "
            f"{parallel_result['Time (sec)']} sec"
        )
    }
])


display(summary)


print("\nExperiments completed successfully.")

EXPERIMENT 1: BAD vs GOOD TOOL DESCRIPTIONS

Tool-selection accuracy:


,Description,Correct,Total Prompts,Accuracy (%)
0,Bad,10,10,100.0
1,Good,10,10,100.0



EXPERIMENT 2: ReAct vs PLAN-AND-EXECUTE

Running: Calculate 125 * 48.

[Tool Attempt]
calculator - attempt 1

Running: What does DocuChat use to store document embeddings?

[Tool Attempt]
document_lookup - attempt 1

[Tool Attempt]
document_lookup - attempt 1

Running: Show me the users in the database.

[Tool Attempt]
mock_db_query - attempt 1

Running: Find information about RAG in the documents.

[Tool Attempt]
document_lookup - attempt 1

Running: List the documents stored in the database.

[Tool Attempt]
mock_db_query - attempt 1

ReAct vs Plan-and-Execute:


,Task,ReAct Steps,ReAct Tokens,ReAct Success,Plan Steps,Plan Tokens,Plan Success
0,Calculate 125 * 48.,2,698,True,0,0,False
1,What does DocuChat use to store document embed...,2,783,True,3,1015,True
2,Show me the users in the database.,2,809,True,0,0,False
3,Find information about RAG in the documents.,2,752,True,0,0,False
4,List the documents stored in the database.,2,820,True,0,0,False



EXPERIMENT 3: TEMPERATURE 0 vs 0.7

All temperature runs:


...


EXPERIMENT 4: PARALLEL TOOL CALLS

[Tool Attempt]
mock_db_query - attempt 1

[Tool Attempt]
mock_db_query - attempt 1

[Tool Attempt]
mock_db_query - attempt 1

[Tool Attempt]
mock_db_query - attempt 1

Sequential vs Parallel:


,Mode,Tool Calls,Time (sec),Success
0,Sequential,2,0.000460,True
1,Parallel,2,0.002041,True



FINAL EXPERIMENT SUMMARY


,Experiment,Measurement,Result
0,Bad vs Good Tool Descriptions,Tool-selection accuracy,Bad: 100.00% | Good: 100.00%
1,ReAct vs Plan-and-Execute,Success on 5 tasks,ReAct: 5/5 | Plan: 1/5
2,Temperature 0 vs 0.7,Consistency over 5 runs,T=0: 100.0% | T=0.7: 100.0%
3,Parallel Tool Calls,Execution time,Sequential: 0.00046 sec | Parallel: 0.002041 sec



Experiments completed successfully.
